In [30]:
import pandas as pd
from sklearn import ensemble
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor

import matplotlib.pyplot as plt

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [11]:
df = pd.read_csv("dataset_mod1.csv", index_col=0)

In [12]:
df.head()

,popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_norm
0,73,0.676,0.4610,-6.746,0.1430,0.0322,0.000001,0.3580,0.715,87.917,0.044043
1,55,0.420,0.1660,-17.235,0.0763,0.9240,0.000006,0.1010,0.267,77.489,0.028566
2,57,0.438,0.3590,-9.734,0.0557,0.2100,0.000000,0.1170,0.120,76.332,0.040255
3,71,0.266,0.0596,-18.515,0.0363,0.9050,0.000071,0.1320,0.143,181.740,0.038557
4,82,0.618,0.4430,-9.681,0.0526,0.4690,0.000000,0.0829,0.167,119.949,0.037969


In [13]:
values = df['popularity']
points = df.drop(['popularity'], axis=1)
train_points, test_points, train_values, test_values = train_test_split(points, values, test_size = 0.2)


In [23]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(train_points, train_values)
rf_predict = rf_model.predict(test_points)
print(mean_absolute_error(test_values, rf_predict))

10.197859500909747


In [31]:
cat = CatBoostRegressor(
    iterations=1200,
    depth=8,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    verbose=0,
    loss_function='MAE'
)

xgb = XGBRegressor(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb = LGBMRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# ====================== 4. Ансамбль моделей (Voting Regressor) ======================
ensemble_model = VotingRegressor(
    estimators=[
        ('LinearRegression', lr_model),
        ('RandomForest', rf_model),
        ('XGBoost', xgb_model)
    ],
    weights=[3, 3, 1]        # даём больший вес XGBoost
)

# ====================== 5. Обучение всех моделей ======================
print("Обучение моделей...")

lr_model.fit(train_points, train_values)
rf_model.fit(train_points, train_values)
xgb_model.fit(train_points, train_values)
ensemble_model.fit(train_points, train_values)

# ====================== 6. Предсказание и оценка ======================
pred_lr = lr_model.predict(test_points)
pred_rf = rf_model.predict(test_points)
pred_xgb = xgb_model.predict(test_points)
pred_ensemble = ensemble_model.predict(test_points)

# ====================== 7. Результаты ======================
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ МОДЕЛЕЙ")
print("="*50)

print(f"Linear Regression    MAE = {mean_absolute_error(test_values, pred_lr):.3f}")
print(f"Random Forest        MAE = {mean_absolute_error(test_values, pred_rf):.3f}")
print(f"XGBoost              MAE = {mean_absolute_error(test_values, pred_xgb):.3f}")
print(f"Ансамбль (Voting)    MAE = {mean_absolute_error(test_values, pred_ensemble):.3f}")

# Дополнительно R²
print(f"\nR² Ансамбля = {r2_score(test_values, pred_ensemble):.4f}")

Обучение моделей...

РЕЗУЛЬТАТЫ МОДЕЛЕЙ
Linear Regression    MAE = 14.032
Random Forest        MAE = 10.198
XGBoost              MAE = 12.052
Ансамбль (Voting)    MAE = 11.896

R² Ансамбля = 0.3353
